# 03 — GPR gate: does geopolitical risk earn its place?

**The quarantine test.** GPR has been kept out of the panels deliberately. It
enters the project only if it adds **incremental** explanatory power over the
factors already carried (VIX + real yields). A lot of what looks like a
"conflict effect" is really just risk-off already captured by VIX. This notebook
answers: once VIX and real yields are in the model, does GPR add anything?

**Test design (honest, not confirmatory):**
1. Baseline: regress commodity returns on VIX change + real-yield change.
2. Augmented: add GPR change (and the threats/acts split).
3. Compare adjusted R² and the GPR coefficient's significance.
4. Check the *calm-regime* subset specifically — GPR's best shot is explaining
   variance when the yield link is weak.

**Discipline:** explore window only. If GPR passes, it graduates to a holdout
check later; it does not get to skip straight into a model.

In [7]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
import config
from src import fetch_gpr

# statsmodels for clean OLS with inference
import statsmodels.api as sm

plt.rcParams["figure.figsize"] = (11, 5)

In [8]:
import sys
!{sys.executable} -m pip install xlrd openpyxl statsmodels

## Load panel + GPR, align on the explore window

In [ ]:
import importlib
import src.fetch_gpr
importlib.reload(src.fetch_gpr)
from src.fetch_gpr import load_gpr

monthly = pd.read_parquet(config.DATA / "commodities_monthly.parquet")
gpr = load_gpr()
if gpr.empty:
    raise SystemExit("GPR not loaded")
df = monthly.join(gpr, how="left").loc[:config.EXPLORE_END]
print("panel+GPR explore:", df.shape)
print("GPR coverage:", df["gpr"].notna().sum(), "months")
df[["gpr","gpr_threats","gpr_acts"]].describe().round(1)

/Users/carlinpercha/Downloads/commodity-macro-dashboard/src/fetch_gpr.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw["date"] = pd.to_datetime(dv, format="%Y%m", errors="coerce")


SystemExit: GPR not loaded -- see fetch_gpr instructions to place the file.

/usr/local/Caskroom/miniconda/base/envs/gentrify/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# GPR is a level index; use its monthly change (log) as the regressor, matching
# how the other macro factors enter (changes, not levels)
d = pd.DataFrame(index=df.index)
d["gold_ret"]  = np.log(df["gold"]).diff()
d["oil_ret"]   = np.log(df["brent"]).diff()
d["vix_chg"]   = df["vix"].diff()
d["real_chg"]  = df["ust10y_real"].diff()
d["gpr_chg"]   = np.log(df["gpr"]).diff()
d["gprt_chg"]  = np.log(df["gpr_threats"]).diff()
d["gpra_chg"]  = np.log(df["gpr_acts"]).diff()
d = d.dropna()
print("regression sample:", d.shape[0], "months")

## The gate — gold

Gold is the cleanest test: it's the commodity most associated with the
safe-haven / geopolitical story, so if GPR fails to add value *here*, it fails
everywhere.

In [ ]:
def compare(y, X_base_cols, X_aug_cols, label):
    y = d[y]
    Xb = sm.add_constant(d[X_base_cols])
    Xa = sm.add_constant(d[X_aug_cols])
    mb = sm.OLS(y, Xb).fit()
    ma = sm.OLS(y, Xa).fit()
    print(f"=== {label} ===")
    print(f"baseline  adj-R2 = {mb.rsquared_adj:.3f}  ({', '.join(X_base_cols)})")
    print(f"augmented adj-R2 = {ma.rsquared_adj:.3f}  (+ {', '.join(set(X_aug_cols)-set(X_base_cols))})")
    print(f"delta adj-R2     = {ma.rsquared_adj - mb.rsquared_adj:+.3f}")
    # GPR coefficient significance in augmented model
    for extra in set(X_aug_cols) - set(X_base_cols):
        print(f"  {extra}: coef={ma.params[extra]:+.4f}  p={ma.pvalues[extra]:.3f}")
    print()
    return mb, ma

_ = compare("gold_ret", ["vix_chg","real_chg"], ["vix_chg","real_chg","gpr_chg"],
            "GOLD: baseline vs +GPR")

**Reading the gate:** if `delta adj-R2` is near zero and the GPR p-value is
> 0.05, GPR adds nothing over VIX + real yields for gold — it's redundant, and
the honest call is to leave it out. If delta is meaningfully positive and GPR is
significant, it earns a place.

In [ ]:
# threats vs acts -- markets often react to anticipation (threats) more than
# realization (acts). Test them separately.
_ = compare("gold_ret", ["vix_chg","real_chg"],
            ["vix_chg","real_chg","gprt_chg","gpra_chg"],
            "GOLD: baseline vs +threats +acts")

## The gate — oil (the channel where conflict *should* matter most)

In [ ]:
_ = compare("oil_ret", ["vix_chg","real_chg"], ["vix_chg","real_chg","gpr_chg"],
            "OIL: baseline vs +GPR")

## Calm-regime subset — GPR's best shot

Earlier we found the gold-real-yield link weakens in calm markets. If GPR
explains gold when yields don't, that residual value would show up specifically
in the calm regime. Rebuild the regime flag and re-run the gate on calm months.

In [ ]:
core_prices = ["gold","silver","wti","brent","natgas","copper","palladium","platinum"]
core_ret = np.log(df[core_prices].where(df[core_prices]>0)).diff()
core_vol = core_ret.rolling(12).std().mean(axis=1)
calm = (core_vol <= core_vol.quantile(0.70))
dc = d.join(calm.rename("calm")).dropna()
dcalm = dc[dc["calm"]]
print("calm months in regression sample:", len(dcalm))

yb = dcalm["gold_ret"]
Xb = sm.add_constant(dcalm[["vix_chg","real_chg"]])
Xa = sm.add_constant(dcalm[["vix_chg","real_chg","gpr_chg"]])
mb = sm.OLS(yb, Xb).fit(); ma = sm.OLS(yb, Xa).fit()
print(f"CALM baseline  adj-R2 = {mb.rsquared_adj:.3f}")
print(f"CALM augmented adj-R2 = {ma.rsquared_adj:.3f}")
print(f"CALM delta     = {ma.rsquared_adj-mb.rsquared_adj:+.3f}")
print(f"  gpr_chg: coef={ma.params['gpr_chg']:+.4f}  p={ma.pvalues['gpr_chg']:.3f}")

## Verdict

Fill in after running:
- **Gold, full sample:** GPR incremental? (delta R², p-value)
- **Threats vs acts:** does either sub-index beat the combined?
- **Oil:** stronger channel than gold?
- **Calm regime:** does GPR rescue the weak-yield periods?

**Decision rule:** GPR graduates into the panels only if it's significant and
adds meaningful adj-R² in at least one clean case. Otherwise it stays out and
the writeup notes that geopolitical risk is subsumed by VIX + real yields at
monthly frequency — itself a defensible, publishable finding.